In [1]:
import pandas as pd

# Load data
clover = pd.read_csv("clover_bat_cov_subset.csv", dtype=str)
bat_matrix = pd.read_csv("bat_virus_interaction_matrix_formatted.csv", dtype=str)
tree_matrix = pd.read_csv("interaction_matrix_tree_leaves.csv", dtype=str)

# --- Step 1. Build mapping from host name -> HostTaxID ---
# Normalize host names to snake_case like in bat_matrix
def normalize_name(name):
    return name.strip().lower().replace(" ", "_")

clover["host_norm"] = clover["Host"].apply(normalize_name)
host_map = dict(zip(clover["host_norm"], clover["HostTaxID"]))

print("Host mapping examples:", list(host_map.items())[:5])

# --- Step 2. Rename bat_matrix host columns to host_<HostTaxID> ---
bat_matrix = bat_matrix.set_index("Virus_fmt")

new_cols = []
for col in bat_matrix.columns:
    if col in host_map:
        new_cols.append(f"host_{host_map[col]}")
    else:
        new_cols.append(col)  # keep virus_fmt or unmapped
bat_matrix.columns = new_cols

# --- Step 3. Align both matrices on common viruses & hosts ---
bat_matrix = bat_matrix.apply(pd.to_numeric, errors="ignore")
tree_matrix = tree_matrix.set_index(tree_matrix.columns[0]).apply(pd.to_numeric, errors="ignore")

common_viruses = bat_matrix.index.intersection(tree_matrix.index)
common_hosts = bat_matrix.columns.intersection(tree_matrix.columns)

print(f"Common viruses: {len(common_viruses)}")
print(f"Common hosts: {len(common_hosts)}")

bat_sub = bat_matrix.loc[common_viruses, common_hosts]
tree_sub = tree_matrix.loc[common_viruses, common_hosts]

# --- Step 4. Count overlaps ---
overlap = (bat_sub & tree_sub)
num_edges_clover = bat_sub.values.sum()
num_edges_tree = tree_sub.values.sum()
num_overlap = overlap.values.sum()

print(f"Edges from Clover matrix: {num_edges_clover}")
print(f"Edges from Tree matrix: {num_edges_tree}")
print(f"Overlapping edges: {num_overlap}")

# Save explicit overlap pairs
overlap_edges = [
    (virus, host)
    for virus, row in overlap.iterrows()
    for host, val in row.items() if val == 1
]
pd.DataFrame(overlap_edges, columns=["Virus","Host"]).to_csv("overlap_edges.csv", index=False)
print("Saved overlap edges to overlap_edges.csv")


Host mapping examples: [('chaerephon_plicatus', '478698'), ('hipposideros_abae', '249020'), ('hipposideros_caffer', '302402'), ('hipposideros_crumeniferus', nan), ('miniopterus_magnater', '438766')]
Common viruses: 9
Common hosts: 19
Edges from Clover matrix: 27
Edges from Tree matrix: 30
Overlapping edges: 27
Saved overlap edges to overlap_edges.csv


/var/folders/81/_4qkc5w17kd767vqrndc8k440000gp/T/ipykernel_1348/35034691.py:30: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  bat_matrix = bat_matrix.apply(pd.to_numeric, errors="ignore")
/var/folders/81/_4qkc5w17kd767vqrndc8k440000gp/T/ipykernel_1348/35034691.py:31: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  tree_matrix = tree_matrix.set_index(tree_matrix.columns[0]).apply(pd.to_numeric, errors="ignore")
